# Q04: VRAM Requirements & KV Caching
**Topic:** LLM | **Difficulty:** Hard | **Time:** 30 min
**Sheet:** GrindLLM50

---

## Question
How much VRAM is required to load a 7B LLM? How do you optimize inference via KV Caching?

## Detailed Answer

### VRAM for Loading a 7B Model

**Model weights only:**
| Precision | Bytes/param | VRAM (7B params) |
|-----------|------------|------------------|
| FP32 | 4 | 28 GB |
| FP16/BF16 | 2 | 14 GB |
| INT8 | 1 | 7 GB |
| INT4 | 0.5 | 3.5 GB |

**Additional VRAM during inference:**
- KV Cache: depends on context length, batch size
- Activations: typically 1-2 GB
- Framework overhead: ~1 GB
- **Rule of thumb**: 1.2× model size for inference

### KV Cache

**The Problem:**
During autoregressive generation, each new token requires attending to ALL previous tokens. Without caching, you'd recompute K and V for all previous positions at each step — O(n²) total work.

**The Solution:**
Store K and V from previous positions. For each new token:
1. Compute Q, K, V for the new token only
2. Append K, V to the cache
3. Attend new Q to entire K cache
4. Return output + updated cache

**KV Cache Size:**
$$\text{KV Cache} = 2 \times n_\text{layers} \times n_\text{heads} \times d_\text{head} \times n_\text{seq} \times b \times \text{bytes}$$

For LLaMA-7B (FP16, seq=4096, batch=1):
$$2 \times 32 \times 32 \times 128 \times 4096 \times 2 \text{ bytes} \approx 2 \text{ GB}$$

### KV Cache Optimizations
| Technique | How | Savings |
|-----------|-----|--------|
| **GQA** | Share KV across head groups | 4-8× less KV cache |
| **MQA** | Single KV for all heads | n_heads × less cache |
| **Quantized KV** | INT8 KV cache | 2× |
| **PagedAttention** | Paged memory allocation (vLLM) | Better utilization |
| **Sliding window** | Only cache last W tokens | W/n × less |

## Key Takeaways
- 7B model FP16 = **14 GB VRAM** (weights only), ~16-17 GB practical
- KV cache avoids recomputing K,V for past tokens — essential for fast generation
- KV cache is the **primary memory bottleneck** for long contexts
- **GQA + quantized KV + PagedAttention** is the modern optimization stack